# 02 · The linear base — enetreg2 → linbest (the locked floor)

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first.** Each QLIKE is a cluster walk-forward `resid_amortized` base fit (full-OOS, deferred
Duan smearing). The pipeline machinery is the through-line notebook; here we fold the **thin base-form
sequence** and the FWL attribution that says *why* the base is at its floor.

In [ ]:
import html, inspect, json, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

def _details(f, open_=False):
    mod = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(mod + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f):
    return Markdown(_details(f))
print("setup ok")

---
## 1 · Verify — the base evolution

`enetreg2` = HAR×{open,close} + cumrv×close (fixed-α single pass). Four **orthogonal** improvers, each
downstream-subsumed by the tree, **stack** to the locked linear floor `linbest`.

In [ ]:
# cluster-established base-form QLIKE (writeup mechanism_and_data_to_buy §8); provenance = cluster collect
base = pd.DataFrame([
    ("plain enet",                       0.12516, "—"),
    ("enetreg2 (HAR×{open,close}+cumrv×close)", 0.12314, "base"),
    ("+ har5rank (rank-space close HAR-damp)",  0.12293, "-0.00021"),
    ("+ ivmag (VIX magnitude, not rank)",       0.12301, "-0.00013"),
    ("+ sig (vol-arrival Bowley path moments)", 0.12302, "-0.00012"),
    ("+ exogrel (long-window rolling-rel exog)",0.12305, "-0.00009"),
    ("linbest (all 4 stacked)",                 0.12266, "-0.00048"),
], columns=["base form", "qlike", "Δ vs enetreg2 (isolated)"])
display(base)
assert abs(base.qlike.min() - 0.12266) < 1e-9  # linbest is the floor
print("linbest = 0.12266 — the locked linear foundation the DL residual builds on (ch. 04).")

## 2 · Interpret — why the base is at its floor (FWL block attribution)

Type-III unique contributions of the linear base (`fwl_attribution.py`; full fit reproduces 0.12314):

| block | unique ΔQLIKE | note |
|---|---|---|
| HAR (6) | **−0.02208** | 96% of explainable variance (R² 0.594 of 0.617) |
| EXOG (359) | −0.00449 | the whole raw-exog panel |
| REGIME (HAR×{open,close}) | −0.00225 | the session-edge interaction |
| CUMRV (1) | −0.00122 | **~27× more value per feature than the 359 exog** |

- **`cumrv` is ~27× more valuable per feature than the raw exog** — direct measurement of the mechanism
  dominates piling on indirect series (the data-to-buy case).
- The improvers **stack to 87%** of their orthogonal sum (−0.00048 vs −0.00055) — near-additive.
- But **`linbest` is a legible base, not a QLIKE lever**: its −0.00048 linear edge **collapses to −0.00002
  at the +EBM stage** — the tree/EBM subsume it. So linbest is the *clean residual* for the DL stage, and
  the deployed 0.12035 was already at the floor. The penalty is **not basis-invariant** (unpenalizing the
  dense HAR block is neutral + legible); a global α can't price sparse close-gated columns (ch. 03).